# Repository guide: MMS half-SPK009; SpecAugment

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# MMS-1B Tarifit V1.2 — Half-Bible Data-Balance Experiment

Controlled ablation of training-data composition.

- Keep 100% of SPK001 and SPK002.
- Keep about 50% of SPK009 Bible **duration within every recording** using deterministic seed 42.
- Keep the same frozen 129-segment validation set.
- Re-run the strongest MMS setup: adapter-only `facebook/mms-1b-all` + SpecAugment, 4 epochs.

Only the training-data composition changes. The goal is to test whether reducing one-speaker/read-domain dominance improves generalization despite fewer training hours.


In [ ]:
# Cell 1 — Install exact dependencies
!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"
print("✓ Dependencies installed. Restart runtime once if Colab requests it.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
✓ Dependencies installed. Restart runtime once if Colab requests it.


In [ ]:
# Cell 2 — Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
PROJECT_ROOT=Path("/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm")
METADATA_PATH=PROJECT_ROOT/"data/metadata/segments_metadata_v1_2.csv"
FROZEN_METADATA_PATH=PROJECT_ROOT/"data/metadata/segments_metadata_v1_2_train_val_frozen.csv"
TOKENIZER_DIR=PROJECT_ROOT/"data/processed/mms_tokenizer_v1_2"
DATASET_CACHE_DIR=PROJECT_ROOT/"data/processed/mms_corpus_v1_2"
CACHE_MANIFEST_PATH=DATASET_CACHE_DIR/"cache_manifest.json"
OUTPUT_DIR=PROJECT_ROOT/"models/mms_1b_tarifit_v1_2_half_bible_specaug"
RESULTS_DIR=PROJECT_ROOT/"results/mms_1b_tarifit_v1_2_half_bible_specaug"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True); RESULTS_DIR.mkdir(parents=True,exist_ok=True)
SUBSET_METADATA_PATH=RESULTS_DIR/"half_bible_training_subset.csv"
SUBSET_IDS_PATH=RESULTS_DIR/"half_bible_training_segment_ids.txt"
BASE_MODEL="facebook/mms-1b-all"
print(PROJECT_ROOT); print(OUTPUT_DIR)


Mounted at /content/drive
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug


In [ ]:
# Cell 3 — Verify versions, GPU, and seed
import sys,json,random,hashlib,numpy as np,pandas as pd,torch,transformers,datasets
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Transformers:",transformers.__version__); print("Datasets:",datasets.__version__)
assert transformers.__version__=="4.57.1" and datasets.__version__=="4.4.1"
assert torch.cuda.is_available(), "Switch Colab to GPU"
print("GPU:",torch.cuda.get_device_name(0))


Transformers: 4.57.1
Datasets: 4.4.1
GPU: Tesla T4


In [ ]:
# Cell 4 — Load the exact frozen V1.2 train/validation split
frozen_df=pd.read_csv(FROZEN_METADATA_PATH)
for c in ["segment_id","recording_id","speaker_group_id","dataset_split","transcription"]:
    frozen_df[c]=frozen_df[c].fillna("").astype(str).str.strip()
frozen_df["dataset_split"]=frozen_df["dataset_split"].str.lower()
frozen_df["duration_seconds"]=pd.to_numeric(frozen_df["duration_seconds"],errors="raise")
full_train_df=frozen_df[frozen_df.dataset_split.eq("train")].copy()
val_df=frozen_df[frozen_df.dataset_split.eq("validation")].copy()
assert len(full_train_df)==1754 and len(val_df)==129
print("Full train:",len(full_train_df),round(full_train_df.duration_seconds.sum()/3600,3),"h")
print("Validation:",len(val_df),round(val_df.duration_seconds.sum()/3600,3),"h")
display(full_train_df.groupby("speaker_group_id").agg(segments=("segment_id","count"),hours=("duration_seconds",lambda x:x.sum()/3600),recordings=("recording_id","nunique")).reset_index())


Full train: 1754 5.223 h
Validation: 129 0.298 h


,speaker_group_id,segments,hours,recordings
0,SPK001,96,0.237127,4
1,SPK002,186,0.347604,4
2,SPK009,1472,4.638002,44


In [ ]:
# Cell 5 — Split Bible speaker from the other training speakers
bible_full_df=full_train_df[full_train_df.speaker_group_id.eq("SPK009")].copy()
other_train_df=full_train_df[~full_train_df.speaker_group_id.eq("SPK009")].copy()
assert len(bible_full_df)>0
print("Bible SPK009:",len(bible_full_df),round(bible_full_df.duration_seconds.sum()/3600,3),"h")
print("Other speakers:",len(other_train_df),round(other_train_df.duration_seconds.sum()/3600,3),"h")


Bible SPK009: 1472 4.638 h
Other speakers: 282 0.585 h


In [ ]:
# Cell 6 — Deterministically retain about 50% of Bible duration inside every recording
BIBLE_KEEP_FRACTION=0.50
kept_parts=[]; audit=[]
for recording_id,group in bible_full_df.groupby("recording_id",sort=True):
    stable_seed=int(hashlib.sha256(f"{SEED}:{recording_id}".encode()).hexdigest()[:8],16)
    rng=np.random.default_rng(stable_seed)
    shuffled=group.iloc[rng.permutation(len(group))].copy()
    total=float(group.duration_seconds.sum()); target=0.5*total
    cumulative=shuffled.duration_seconds.cumsum().to_numpy()
    cutoff=min(int(np.searchsorted(cumulative,target,side="left")),len(shuffled)-1)
    kept=shuffled.iloc[:cutoff+1].copy(); kept_parts.append(kept)
    audit.append({"recording_id":recording_id,"full_segments":len(group),"kept_segments":len(kept),"full_seconds":total,"target_seconds":target,"kept_seconds":float(kept.duration_seconds.sum()),"retained_pct":100*float(kept.duration_seconds.sum())/total})
bible_half_df=pd.concat(kept_parts,ignore_index=True)
bible_audit=pd.DataFrame(audit)
assert set(bible_half_df.recording_id)==set(bible_full_df.recording_id)
display(bible_audit)
print("Bible retained:",len(bible_half_df),round(bible_half_df.duration_seconds.sum()/3600,3),"h")
print("Actual retained duration: %.2f%%"%(100*bible_half_df.duration_seconds.sum()/bible_full_df.duration_seconds.sum()))


,recording_id,full_segments,kept_segments,full_seconds,target_seconds,kept_seconds,retained_pct
0,REC094,19,10,221.328,110.6640,127.904,57.789344
1,REC095,25,13,274.784,137.3920,146.960,53.482008
2,REC096,16,10,187.030,93.5150,106.702,57.050741
3,REC097,22,11,258.385,129.1925,132.168,51.151576
4,REC098,43,22,486.318,243.1590,250.668,51.544051
5,REC099,34,17,362.224,181.1120,196.176,54.158753
6,REC100,23,11,267.850,133.9250,140.500,52.454732
7,REC101,29,16,327.072,163.5360,167.976,51.357499
8,REC102,29,14,334.294,167.1470,167.604,50.136706
9,REC103,33,17,407.965,203.9825,206.936,50.723959


Bible retained: 759 2.406 h
Actual retained duration: 51.88%


In [ ]:
# Cell 7 — Build and freeze the half-Bible training subset

half_train_df = pd.concat(
    [
        other_train_df,
        bible_half_df,
    ],
    ignore_index=True,
)

# Stable ordering for reproducibility
half_train_df = (
    half_train_df
    .sort_values(
        ["speaker_group_id", "recording_id", "segment_id"]
    )
    .reset_index(drop=True)
)

assert half_train_df["segment_id"].is_unique

# Keep all non-Bible training data
assert set(other_train_df["segment_id"]) <= set(
    half_train_df["segment_id"]
)

# Validation must remain completely separate
assert not (
    set(half_train_df["segment_id"])
    & set(val_df["segment_id"])
)

# Save the selected subset metadata
half_train_df.to_csv(
    SUBSET_METADATA_PATH,
    index=False,
    encoding="utf-8",
)

# Save exact selected segment IDs
with open(
    SUBSET_IDS_PATH,
    "w",
    encoding="utf-8",
) as f:
    for segment_id in half_train_df["segment_id"]:
        f.write(f"{segment_id}\n")

# Fingerprint the exact selected training subset
subset_hash = hashlib.sha256(
    "\n".join(
        half_train_df["segment_id"].astype(str).tolist()
    ).encode("utf-8")
).hexdigest()

print("Half-Bible train segments:", len(half_train_df))

print(
    "Half-Bible train hours:",
    round(
        half_train_df["duration_seconds"].sum() / 3600,
        3,
    ),
)

print(
    "Bible retained hours:",
    round(
        bible_half_df["duration_seconds"].sum() / 3600,
        3,
    ),
)

print(
    "Non-Bible hours:",
    round(
        other_train_df["duration_seconds"].sum() / 3600,
        3,
    ),
)

print(
    "Bible share of new training duration:",
    f"{100 * bible_half_df['duration_seconds'].sum() / half_train_df['duration_seconds'].sum():.2f}%"
)

print("Subset SHA256:", subset_hash)

print(
    "Saved subset metadata:",
    SUBSET_METADATA_PATH,
)

print(
    "Saved subset IDs:",
    SUBSET_IDS_PATH,
)

print("✓ Half-Bible training subset frozen successfully.")

Half-Bible train segments: 1041
Half-Bible train hours: 2.991
Bible retained hours: 2.406
Non-Bible hours: 0.585
Bible share of new training duration: 80.45%
Subset SHA256: 1484ba365ec994d250825c8069aa3a3512037615c3a19ad8d07283d0048464bf
Saved subset metadata: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_half_bible_specaug/half_bible_training_subset.csv
Saved subset IDs: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_half_bible_specaug/half_bible_training_segment_ids.txt
✓ Half-Bible training subset frozen successfully.


In [ ]:
# Cell 8 — Compare full and half-Bible corpus composition
rows=[]
for name,frame in [("Full V1.2",full_train_df),("Half-Bible V1.2",half_train_df)]:
    bible_s=frame.loc[frame.speaker_group_id.eq("SPK009"),"duration_seconds"].sum()
    rows.append({"condition":name,"segments":len(frame),"hours":frame.duration_seconds.sum()/3600,"speakers":frame.speaker_group_id.nunique(),"SPK009_hours":bible_s/3600,"non_Bible_hours":(frame.duration_seconds.sum()-bible_s)/3600,"SPK009_duration_pct":100*bible_s/frame.duration_seconds.sum()})
corpus_comparison=pd.DataFrame(rows); display(corpus_comparison)
corpus_comparison.to_csv(RESULTS_DIR/"corpus_comparison.csv",index=False)


,condition,segments,hours,speakers,SPK009_hours,non_Bible_hours,SPK009_duration_pct
0,Full V1.2,1754,5.222733,3,4.638002,0.584731,88.804117
1,Half-Bible V1.2,1041,2.990990,3,2.406259,0.584731,80.450247


In [ ]:
# Cell 9 — Verify speaker independence and held-out test isolation
train_speakers=set(half_train_df.speaker_group_id); val_speakers=set(val_df.speaker_group_id)
print("Train/validation overlap:",bool(train_speakers&val_speakers)); assert not(train_speakers&val_speakers)
master_df=pd.read_csv(METADATA_PATH)
master_df["dataset_split"]=master_df.dataset_split.fillna("").astype(str).str.strip().str.lower()
master_df["speaker_group_id"]=master_df.speaker_group_id.fillna("").astype(str).str.strip()
test_speakers=set(master_df.loc[master_df.dataset_split.eq("test"),"speaker_group_id"])
print("Train/test overlap:",bool(train_speakers&test_speakers)); assert not(train_speakers&test_speakers)
print("Validation/test overlap:",bool(val_speakers&test_speakers)); assert not(val_speakers&test_speakers)
print("✓ No speaker leakage")


Train/validation overlap: False
Train/test overlap: False
Validation/test overlap: False
✓ No speaker leakage


In [ ]:
# Cell 10 — Load the exact existing V1.2 tokenizer
from transformers import Wav2Vec2CTCTokenizer,Wav2Vec2FeatureExtractor,Wav2Vec2Processor
FINAL_LETTERS=["a","b","c","d","ḍ","e","ɛ","f","g","h","ḥ","i","j","k","l","m","n","p","q","r","s","t","ṭ","u","v","w","x","y","z","ɣ","ʷ"]
VOCAB_PATH=TOKENIZER_DIR/"vocab.json"; assert VOCAB_PATH.exists()
existing_vocab=json.loads(VOCAB_PATH.read_text(encoding="utf-8"))
expected=set(FINAL_LETTERS)|{"|","[UNK]","[PAD]"}
assert len(existing_vocab)==34 and set(existing_vocab)==expected and sorted(existing_vocab.values())==list(range(34))
tokenizer=Wav2Vec2CTCTokenizer(vocab_file=str(VOCAB_PATH),unk_token="[UNK]",pad_token="[PAD]",word_delimiter_token="|",bos_token=None,eos_token=None,do_lower_case=False)
feature_extractor=Wav2Vec2FeatureExtractor(feature_size=1,sampling_rate=16000,padding_value=0.0,do_normalize=True,return_attention_mask=True)
processor=Wav2Vec2Processor(feature_extractor=feature_extractor,tokenizer=tokenizer)
print("Tokenizer size:",len(tokenizer)); assert len(tokenizer)==34


Tokenizer size: 34


In [ ]:
# Cell 11 — Reuse the exact frozen cached waveforms and labels
from datasets import load_from_disk,DatasetDict
metadata_sha256=hashlib.sha256(FROZEN_METADATA_PATH.read_bytes()).hexdigest()
manifest=json.loads(CACHE_MANIFEST_PATH.read_text(encoding="utf-8")); assert manifest.get("metadata_sha256")==metadata_sha256
full_dataset=load_from_disk(str(DATASET_CACHE_DIR)); assert len(full_dataset["train"])==1754 and len(full_dataset["validation"])==129
id_to_index={sid:i for i,sid in enumerate(full_dataset["train"]["segment_id"])}
missing=[sid for sid in half_train_df.segment_id if sid not in id_to_index]; assert not missing
half_indices=[id_to_index[sid] for sid in half_train_df.segment_id]
experiment_dataset=DatasetDict({"train":full_dataset["train"].select(half_indices),"validation":full_dataset["validation"]})
assert list(experiment_dataset["train"]["segment_id"])==list(half_train_df.segment_id)
print(experiment_dataset)
print("Cached train hours:",round(sum(experiment_dataset["train"]["input_length"])/16000/3600,3))


DatasetDict({
    train: Dataset({
        features: ['segment_id', 'input_values', 'input_length', 'labels'],
        num_rows: 1041
    })
    validation: Dataset({
        features: ['segment_id', 'input_values', 'input_length', 'labels'],
        num_rows: 129
    })
})
Cached train hours: 2.991


In [ ]:
# Cell 12 — Load MMS-1B-all, initialize adapters, and enable the matched SpecAugment setup
from transformers import Wav2Vec2ForCTC
MASK_TIME_PROB=0.05; MASK_TIME_LENGTH=5; MASK_FEATURE_PROB=0.0
model=Wav2Vec2ForCTC.from_pretrained(BASE_MODEL,vocab_size=len(tokenizer),pad_token_id=tokenizer.pad_token_id,attention_dropout=0.0,hidden_dropout=0.0,feat_proj_dropout=0.0,layerdrop=0.0,ctc_loss_reduction="mean",ctc_zero_infinity=True,ignore_mismatched_sizes=True)
model.init_adapter_layers(); model.freeze_base_model()
for p in model._get_adapters().values(): p.requires_grad=True
for p in model.lm_head.parameters(): p.requires_grad=True
model.config.apply_spec_augment=True; model.config.mask_time_prob=MASK_TIME_PROB; model.config.mask_time_length=MASK_TIME_LENGTH; model.config.mask_feature_prob=MASK_FEATURE_PROB
total_params=sum(p.numel() for p in model.parameters()); trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params:,}"); print(f"Trainable: {trainable_params:,} ({100*trainable_params/total_params:.4f}%)")
print("SpecAugment:",model.config.apply_spec_augment,model.config.mask_time_prob,model.config.mask_time_length)
assert 950_000_000<total_params<980_000_000 and 1_500_000<trainable_params<3_000_000


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([34]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([34, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total: 964,692,130
Trainable: 2,194,722 (0.2275%)
SpecAugment: True 0.05 5


In [ ]:
# Cell 13 — Verify CTC feasibility on the new training subset and unchanged validation set
def min_ctc(labels):
    labels=list(labels); return len(labels)+sum(labels[i]==labels[i-1] for i in range(1,len(labels)))
def find_bad(ds):
    bad=[]
    for i,e in enumerate(ds):
        out=int(model._get_feat_extract_output_lengths(torch.tensor(int(e["input_length"]))).item())
        need=min_ctc(e["labels"])
        if out<need: bad.append({"index":i,"segment_id":e["segment_id"],"frames":out,"min_ctc":need})
    return bad
bad_train=find_bad(experiment_dataset["train"]); bad_val=find_bad(experiment_dataset["validation"])
print("Bad train:",len(bad_train)); print("Bad val:",len(bad_val))
if bad_train: display(pd.DataFrame(bad_train))
if bad_val: display(pd.DataFrame(bad_val))
assert not bad_train and not bad_val
print("✓ CTC feasible")


Bad train: 0
Bad val: 0
✓ CTC feasible


In [ ]:
# Cell 14 — Define dynamic CTC collator and WER/CER
from dataclasses import dataclass
from typing import Any,Union
from jiwer import wer,cer
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool,str]=True
    def __call__(self,features):
        inp=[{"input_values":f["input_values"]} for f in features]; lab=[{"input_ids":f["labels"]} for f in features]
        batch=self.processor.pad(inp,padding=self.padding,return_tensors="pt")
        labels_batch=self.processor.tokenizer.pad(lab,padding=self.padding,return_tensors="pt")
        batch["labels"]=labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1),-100)
        return batch
data_collator=DataCollatorCTCWithPadding(processor)
def compute_metrics(pred):
    pred_ids=np.argmax(pred.predictions,axis=-1); label_ids=pred.label_ids.copy(); label_ids[label_ids==-100]=tokenizer.pad_token_id
    ps=[x.strip() for x in processor.batch_decode(pred_ids)]; rs=[x.strip() for x in processor.batch_decode(label_ids,group_tokens=False)]
    return {"wer":wer(rs,ps),"cer":cer(rs,ps)}
print("✓ Collator and metrics ready")


✓ Collator and metrics ready


In [ ]:
# Cell 15 — Configure the matched 4-epoch MMS SpecAugment training
from transformers import TrainingArguments,Trainer
training_args=TrainingArguments(output_dir=str(OUTPUT_DIR),per_device_train_batch_size=1,gradient_accumulation_steps=8,per_device_eval_batch_size=1,learning_rate=1e-3,num_train_epochs=4,warmup_steps=20,fp16=True,eval_strategy="epoch",save_strategy="epoch",save_total_limit=2,load_best_model_at_end=True,metric_for_best_model="cer",greater_is_better=False,logging_strategy="steps",logging_steps=25,report_to="none",seed=SEED,data_seed=SEED,dataloader_num_workers=0,remove_unused_columns=False)
trainer=Trainer(model=model,args=training_args,train_dataset=experiment_dataset["train"],eval_dataset=experiment_dataset["validation"],data_collator=data_collator,compute_metrics=compute_metrics,processing_class=processor)
print("Train:",len(experiment_dataset["train"]),"Validation:",len(experiment_dataset["validation"]))
print("LR:",training_args.learning_rate,"epochs:",training_args.num_train_epochs,"effective batch:",8)


Train: 1041 Validation: 129
LR: 0.001 epochs: 4 effective batch: 8


In [ ]:
# Cell 16 — Start or resume training
from transformers.trainer_utils import get_last_checkpoint
last=get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None
print("Resume checkpoint:",last)
train_result=trainer.train(resume_from_checkpoint=last) if last else trainer.train()
print("Best checkpoint:",trainer.state.best_model_checkpoint)
print("Best validation CER:",trainer.state.best_metric)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Resume checkpoint: None


Epoch,Training Loss,Validation Loss,Wer,Cer
1,0.744000,2.849272,0.894558,0.441032
2,0.658100,3.050890,0.877551,0.438138
3,0.618000,3.075217,0.849915,0.427526
4,0.568900,3.032704,0.847364,0.424954


Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524
Best validation CER: 0.42495377441916554


In [ ]:
# Cell 17 — Save epoch history and evaluate the selected best checkpoint
rows=[]; last_train=None
for log in trainer.state.log_history:
    if "loss" in log and "eval_loss" not in log: last_train=log["loss"]
    if "eval_loss" in log: rows.append({"epoch":log.get("epoch"),"training_loss":last_train,"validation_loss":log.get("eval_loss"),"WER":log.get("eval_wer"),"CER":log.get("eval_cer")})
history_df=pd.DataFrame(rows); display(history_df); history_df.to_csv(RESULTS_DIR/"training_history.csv",index=False)
best_metrics=trainer.evaluate(experiment_dataset["validation"]); best_wer=float(best_metrics["eval_wer"]); best_cer=float(best_metrics["eval_cer"])
print(f"Best WER: {best_wer*100:.2f}%"); print(f"Best CER: {best_cer*100:.2f}%"); print("Checkpoint:",trainer.state.best_model_checkpoint)
BEST_MODEL_DIR=OUTPUT_DIR/"best_model"; trainer.save_model(str(BEST_MODEL_DIR)); processor.save_pretrained(str(BEST_MODEL_DIR))


,epoch,training_loss,validation_loss,WER,CER
0,1.0,0.7440,2.849272,0.894558,0.441032
1,2.0,0.6581,3.050890,0.877551,0.438138
2,3.0,0.6180,3.075217,0.849915,0.427526
3,4.0,0.5689,3.032704,0.847364,0.424954


Best WER: 84.74%
Best CER: 42.50%
Checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524


[]

In [ ]:
# Cell 18 — Save validation predictions and space-insensitive CER diagnostic
out=trainer.predict(experiment_dataset["validation"]); pred_ids=np.argmax(out.predictions,axis=-1); label_ids=out.label_ids.copy(); label_ids[label_ids==-100]=tokenizer.pad_token_id
preds=[x.strip() for x in processor.batch_decode(pred_ids)]; refs=[x.strip() for x in processor.batch_decode(label_ids,group_tokens=False)]
pred_df=pd.DataFrame({"segment_id":experiment_dataset["validation"]["segment_id"],"reference":refs,"prediction":preds})
pred_df["segment_wer"]=[wer(r,p) for r,p in zip(refs,preds)]; pred_df["segment_cer"]=[cer(r,p) for r,p in zip(refs,preds)]; pred_df["segment_cer_no_spaces"]=[cer(r.replace(" ",""),p.replace(" ","")) for r,p in zip(refs,preds)]
cer_no_spaces=cer([r.replace(" ","") for r in refs],[p.replace(" ","") for p in preds])
pred_df.to_csv(RESULTS_DIR/"validation_predictions.csv",index=False,encoding="utf-8"); display(pred_df.head(20))
print("Standard CER: %.2f%%"%(best_cer*100)); print("CER without spaces: %.2f%%"%(cer_no_spaces*100))


,segment_id,reference,prediction,segment_wer,segment_cer,segment_cer_no_spaces
0,REC090_SEG0010,ssalamuɛlikum necc meryem,salamuɛalikum nec meryam,1.000000,0.160000,0.173913
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,aqqay ruxa tnaynuɛ crin sanad ihulanḍa,1.000000,0.200000,0.147059
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,mercex ag misenjjiran usiɣ dzi lmeɣrib umiraqq...,0.800000,0.200000,0.142857
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,umi wsiɣ dda ufix manayenni wadji ca miriraɣar...,0.500000,0.109375,0.078431
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,nec mamci ra dji xti lmeɣrib wadji manayeniufi...,0.818182,0.266667,0.180000
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,necdi lmeɣrib ira ɣari lḥurriya inu ira ɣari i...,0.444444,0.125000,0.119048
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,ḍi lmeɣrim ncin mamciraniɛic,0.833333,0.228571,0.200000
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,ak babadimma waɣanex ca ɣanx caneḥwayej nteggi...,0.750000,0.188235,0.133333
8,REC090_SEG0018,lmuhim wsiɣd,muhim usiɣt,1.000000,0.250000,0.272727
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,neccira ɛemmas wawsiɣ dɣa uruppa usiɣ ddi ṭyar...,0.750000,0.191781,0.142857


Standard CER: 42.50%
CER without spaces: 40.67%


In [ ]:
# Cell 19 — Compare directly with the full-data MMS experiments
comparison=pd.DataFrame([
{"experiment":"Full V1.2 — no augmentation","WER":0.866497,"CER":0.434199},
{"experiment":"Full V1.2 — SpecAugment","WER":0.849915,"CER":0.432671},
{"experiment":"Full V1.2 — SpecAugment + speed","WER":0.856293,"CER":0.434360},
{"experiment":"Half-Bible V1.2 — SpecAugment","WER":best_wer,"CER":best_cer},])
comparison["WER_percent"]=100*comparison.WER; comparison["CER_percent"]=100*comparison.CER
display(comparison)
print("Δ WER vs full SpecAugment: %+.2f points"%(100*(best_wer-0.849915)))
print("Δ CER vs full SpecAugment: %+.2f points"%(100*(best_cer-0.432671)))
comparison.to_csv(RESULTS_DIR/"validation_comparison.csv",index=False)


,experiment,WER,CER,WER_percent,CER_percent
0,Full V1.2 — no augmentation,0.866497,0.434199,86.649700,43.419900
1,Full V1.2 — SpecAugment,0.849915,0.432671,84.991500,43.267100
2,Full V1.2 — SpecAugment + speed,0.856293,0.434360,85.629300,43.436000
3,Half-Bible V1.2 — SpecAugment,0.847364,0.424954,84.736395,42.495377


Δ WER vs full SpecAugment: -0.26 points
Δ CER vs full SpecAugment: -0.77 points


In [ ]:
# Cell 20 — Save full experiment summary
summary={"experiment":"MMS-1B Tarifit V1.2 half-Bible + SpecAugment","base_model":BASE_MODEL,"seed":SEED,"full_train_segments":len(full_train_df),"full_train_hours":float(full_train_df.duration_seconds.sum()/3600),"half_bible_train_segments":len(half_train_df),"half_bible_train_hours":float(half_train_df.duration_seconds.sum()/3600),"full_bible_hours":float(bible_full_df.duration_seconds.sum()/3600),"retained_bible_hours":float(bible_half_df.duration_seconds.sum()/3600),"retained_bible_duration_fraction":float(bible_half_df.duration_seconds.sum()/bible_full_df.duration_seconds.sum()),"validation_segments":129,"training_subset_sha256":subset_hash,"sampling":"deterministic recording-stratified duration sampling of SPK009 at ~50% per recording","specaugment":{"mask_time_prob":0.05,"mask_time_length":5,"mask_feature_prob":0.0},"learning_rate":1e-3,"epochs":4,"effective_batch_size":8,"checkpoint_selection_metric":"CER","best_checkpoint":trainer.state.best_model_checkpoint,"best_validation_wer":best_wer,"best_validation_cer":best_cer,"space_insensitive_validation_cer":float(cer_no_spaces),"full_specaugment_reference_wer":0.849915,"full_specaugment_reference_cer":0.432671,"delta_wer_vs_full_specaugment":float(best_wer-0.849915),"delta_cer_vs_full_specaugment":float(best_cer-0.432671)}
(RESULTS_DIR/"experiment_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "experiment": "MMS-1B Tarifit V1.2 half-Bible + SpecAugment",
  "base_model": "facebook/mms-1b-all",
  "seed": 42,
  "full_train_segments": 1754,
  "full_train_hours": 5.222733333333333,
  "half_bible_train_segments": 1041,
  "half_bible_train_hours": 2.990989722222222,
  "full_bible_hours": 4.638002222222221,
  "retained_bible_hours": 2.406258611111111,
  "retained_bible_duration_fraction": 0.5188135959879278,
  "validation_segments": 129,
  "training_subset_sha256": "1484ba365ec994d250825c8069aa3a3512037615c3a19ad8d07283d0048464bf",
  "sampling": "deterministic recording-stratified duration sampling of SPK009 at ~50% per recording",
  "specaugment": {
    "mask_time_prob": 0.05,
    "mask_time_length": 5,
    "mask_feature_prob": 0.0
  },
  "learning_rate": 0.001,
  "epochs": 4,
  "effective_batch_size": 8,
  "checkpoint_selection_metric": "CER",
  "best_checkpoint": "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkp

## Interpretation rule

Compare the half-Bible run primarily with **Full V1.2 + SpecAugment**, because the model, augmentation, tokenizer, validation set, seed, optimizer, and epoch count are held constant.

- Better: reducing dominant-speaker/domain exposure improved generalization enough to offset fewer hours.
- Similar: much of the extra Bible material may be redundant for validation generalization.
- Worse: the additional Bible speech still contributes useful acoustic/linguistic coverage despite the imbalance.

This single 50% ablation does **not** establish an optimal Bible proportion.


In [ ]:
Reducing the contribution of the dominant Bible speaker/domain by approximately half led to a modest improvement in validation performance. The best CER decreased from 43.27% to 42.50%, while WER decreased from 84.99% to 84.74%. This suggests that corpus balance may be at least as important as raw training duration in this low-resource setting, although the improvement is limited and does not establish that a 50% reduction is optimal.